# 🇧🇩 Fine-Tuning BanglaBERT on Real-World Benchmark Dataset (SentNoB)
**Dataset:** `khondoker/SentNoB` (15,000+ authentic, diverse Bengali social media & customer feedback comments)
**Base Model:** `csebuetnlp/banglabert` (ELECTRA Architecture pre-trained on 40GB+ Bengali corpora)
**Target Task:** Customer Sentiment Classification (`POSITIVE` vs `NEGATIVE`)

> 💡 **Note:** You do NOT need to upload any CSV file in Kaggle Input! Hugging Face automatically downloads this dataset when Internet is ON.

## ⚙️ Step 1: Install Required Libraries & Verify GPU
We install Hugging Face `transformers`, `datasets`, `evaluate`, `accelerate`, and `scikit-learn`.

In [ ]:
# Install dependencies quietly
!pip install -q transformers datasets evaluate scikit-learn accelerate seaborn matplotlib

import torch
print("=== Hardware Verification ===")
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: GPU not detected! Go to Kaggle right panel -> Settings -> Accelerator -> GPU T4 x2")

## 📦 Step 2: Load the Authentic `SentNoB` Dataset
We load `khondoker/SentNoB` directly from Hugging Face Hub. It comes with pre-split `train`, `validation`, and `test` sets.

In [ ]:
from datasets import load_dataset

print("[*] Downloading & Loading 'khondoker/SentNoB' from HuggingFace...")
raw_dataset = load_dataset("khondoker/SentNoB")
print(raw_dataset)

print("\n[*] Inspecting 3 samples from train set:")
for i in range(3):
    sample = raw_dataset["train"][i]
    print(f"  Sample {i+1}: Text = '{sample['text']}', Original Label = {sample['label']}")

## 🎯 Step 3: Align Labels for Customer Feedback Sentiment Engine
In SentNoB:
- Label `0` = Neutral
- Label `1` = Positive
- Label `2` = Negative

Our Customer Feedback Engine uses:
- **`NEGATIVE (0)`**: Customer pain points, bugs, poor service (triggers Slack alerts) -> mapped from SentNoB `2`
- **`POSITIVE (1)`**: Customer appreciation, positive experience -> mapped from SentNoB `1`

We filter out Neutral (`0`) to train a high-precision binary sentiment classifier.

In [ ]:
# Filter out Neutral (0) and map Negative (2 -> 0), Positive (1 -> 1)
def remap_labels(example):
    # 2 -> 0 (NEGATIVE), 1 -> 1 (POSITIVE)
    mapped_val = 0 if example["label"] == 2 else 1
    return {"label": mapped_val}

# 1. Filter out Neutral (label 0)
binary_dataset = raw_dataset.filter(lambda x: x["label"] in [1, 2])

# 2. Remap labels
binary_dataset = binary_dataset.map(remap_labels)

print("=== Filtered Dataset Distribution ===")
for split in ["train", "validation", "test"]:
    labels = binary_dataset[split]["label"]
    pos = sum(1 for l in labels if l == 1)
    neg = sum(1 for l in labels if l == 0)
    print(f"{split:12s}: Total {len(labels):,} | POSITIVE (1): {pos:,} | NEGATIVE (0): {neg:,}")

## 🧹 Step 4: Text Normalization
Clean excessive whitespaces, HTML tags, and normalize repeated Bengali punctuation marks.

In [ ]:
import re

def clean_bangla(example):
    t = example.get("text", "") or ""
    # Normalize multiple spaces & newlines
    t = re.sub(r"\s+", " ", t).strip()
    # Strip HTML tags
    t = re.sub(r"<[^>]+>", "", t)
    # Normalize repeated punctuation (e.g. !!!!! -> !)
    t = re.sub(r"([!?.।])\1+", r"\1", t)
    return {"text": t}

cleaned_dataset = binary_dataset.map(clean_bangla)

## 🔤 Step 5: BanglaBERT Tokenization
We load `csebuetnlp/banglabert` tokenizer and tokenize all text with `truncation=True` and `max_length=128`.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "csebuetnlp/banglabert"
print(f"[*] Loading Tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

print("[*] Tokenizing datasets (train / val / test)...")
tokenized_datasets = cleaned_dataset.map(tokenize_fn, batched=True)
print("[*] Tokenization complete!")

## 🏗️ Step 6: Model Architecture & Trainer Configuration
We attach a 2-label classification head, setup evaluation metrics (Accuracy, Precision, Recall, Macro-F1), and configure mixed precision (`fp16=True`) training.

In [ ]:
import inspect
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

print(f"[*] Initializing Base Model: {MODEL_NAME}")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": float(acc),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall)
    }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

OUTPUT_DIR = "/kaggle/working/bangla_sentiment_model"
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,          # Automatically load best model checkpoint
    metric_for_best_model="f1",           # Select model with highest Macro F1
    fp16=torch.cuda.is_available(),       # Mixed precision on GPU
    logging_steps=50,
    seed=42,
    report_to="none"
)

# Compatible with both newer (>=4.46) and older transformers versions
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_datasets["train"],
    "eval_dataset": tokenized_datasets["validation"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)
print("[*] Trainer successfully initialized and ready to train!")

## 🚀 Step 7: Train the Model
Execute fine-tuning on the genuine SentNoB dataset. Watch Training and Validation loss decrease smoothly across epochs.

In [ ]:
# Execute training
trainer.train()

## 📊 Step 8: Evaluate on Test Set & Plot Confusion Matrix
We evaluate the best model checkpoint on the test set that the model never saw during training.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Evaluate on official test set
test_metrics = trainer.evaluate(tokenized_datasets["test"])
print("=== Official Test Set Metrics ===")
for k, v in test_metrics.items():
    print(f"{k:20s}: {v:.4f}" if isinstance(v, float) else f"{k:20s}: {v}")

# Predict on test set
preds_raw = trainer.predict(tokenized_datasets["test"])
y_pred = np.argmax(preds_raw.predictions, axis=-1)
y_true = tokenized_datasets["test"]["label"]

print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=["NEGATIVE", "POSITIVE"]))

# Plot Confusion Matrix Heatmap
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["NEGATIVE", "POSITIVE"],
            yticklabels=["NEGATIVE", "POSITIVE"])
plt.title("Confusion Matrix (Real SentNoB Benchmark)")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")
plt.savefig("/kaggle/working/confusion_matrix.png", bbox_inches="tight")
plt.show()
print("[*] Saved confusion matrix image to: /kaggle/working/confusion_matrix.png")

## ⚔️ Step 9: Real-World Benchmark (Old DistilBERT vs Fine-Tuned BanglaBERT)
Side-by-side comparison on authentic Bengali customer feedback sentences including tricky negation and slang.

In [ ]:
from transformers import pipeline

print("[*] Loading Old Baseline Model (DistilBERT SST-2 English)...")
old_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

print("[*] Loading Newly Fine-Tuned BanglaBERT Model...")
new_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

sample_reviews = [
    "আপনাদের প্রোডাক্টটা অনেক পছন্দ হয়েছে, ডেলিভারিও খুব ফাস্ট ছিল!",
    "একদম ফালতু কোয়ালিটি, টাকাটাই জলে গেল। রিফান্ড চাই।",
    "সার্ভিসটা খারাপ না, তবে ডেলিভারি ম্যানের ব্যবহার আরও ভালো হতে পারত।",
    "পণ্যটি দেখতে যেমন ভালো, ব্যবহারেও তেমনই দারুণ।",
    "আমি অর্ডার করেছিলাম ব্ল্যাক কালার, দিয়ে গেল রেড! চরম হতাশ।"
]

print("\n" + "="*80)
print(f"{'Real Bengali Customer Feedback':<48} | {'Old DistilBERT':<14} | {'New BanglaBERT':<14}")
print("="*80)
for text in sample_reviews:
    old_res = old_pipeline(text)[0]
    new_res = new_pipeline(text)[0]
    old_str = f"{old_res['label'][:3]} ({old_res['score']:.2f})"
    new_str = f"{new_res['label'][:3]} ({new_res['score']:.2f})"
    print(f"{text:<48} | {old_str:<14} | {new_str:<14}")
print("="*80)

## 💾 Step 10: Export Model Artifacts & Generate Download Zip

In [ ]:
# 1. Save best model and tokenizer locally
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"[*] Best model & tokenizer saved to: {OUTPUT_DIR}")

# 2. Package into zip file for easy download
!zip -r /kaggle/working/bangla_sentiment_model.zip /kaggle/working/bangla_sentiment_model
print("\n🎉 COMPLETED! Download 'bangla_sentiment_model.zip' from Kaggle Output panel on the right.")